In [22]:
import pandas as pd
import numpy as np
from huggingface_hub import HfFileSystem
from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

HF_REPO = "vngclinh/goodreads-preprocessed"
hf_fs = HfFileSystem(token=HF_TOKEN)
def hf_read(path):
    with hf_fs.open(f"datasets/{HF_REPO}/{path}", "rb") as f:
        return pd.read_parquet(f)
# user genre theo từng CT
#F1
# user_genre   = hf_read("graph/user_genre_affinity.parquet")  # (233k, 11)

# F2 → đổi thành:
# user_genre = hf_read("graph/user_genre_affinity_f2_rating_only.parquet")

# # F3:
# user_genre = hf_read("graph/user_genre_affinity_f3_rating_length.parquet")

# # F4:
# user_genre = hf_read("graph/user_genre_affinity_f4_rating_votes_length.parquet")

# # F5:
user_genre = hf_read("graph/user_genre_affinity_f5_binary_positive.parquet")


genre_fp     = hf_read("lda/genre_fingerprint.parquet")      # (10, 41)
book_vecs    = hf_read("lda/book_topic_vectors.parquet")     # (447k, 41)

genre_cols = [c for c in user_genre.columns if c != "user_id"]
topic_cols = [c for c in genre_fp.columns if c != "primary_genre"]

print(user_genre.shape, genre_fp.shape, book_vecs.shape)

(219385, 11) (10, 40) (447813, 41)


In [23]:
genre_name_map = {
    "genre_children": "children",
    "genre_comics_graphic": "comics, graphic",
    "genre_fantasy_paranormal": "fantasy, paranormal",
    "genre_fiction": "fiction",
    "genre_history_historical_fiction_biography": "history, historical fiction, biography",
    "genre_mystery_thriller_crime": "mystery, thriller, crime",
    "genre_non-fiction": "non-fiction",
    "genre_poetry": "poetry",
    "genre_romance": "romance",
    "genre_young-adult": "young-adult"
}

U = user_genre[genre_cols].values
G = genre_fp.loc[[genre_name_map[c] for c in genre_cols]].values

user_topic = U @ G
print("user_topic shape:", user_topic.shape)  # expect (233321, 40)

user_topic shape: (219385, 40)


In [24]:
user_topic_df = pd.DataFrame(user_topic, columns=topic_cols)
user_topic_df.insert(0, "user_id", user_genre["user_id"].values)

user_taste = pd.concat([
    user_genre.reset_index(drop=True),
    user_topic_df[topic_cols].reset_index(drop=True)
], axis=1)

print("user_taste_profile:", user_taste.shape)  # expect (233321, 51)
print(user_taste.columns.tolist())

user_taste_profile: (219385, 51)
['user_id', 'genre_children', 'genre_comics_graphic', 'genre_fantasy_paranormal', 'genre_fiction', 'genre_history_historical_fiction_biography', 'genre_mystery_thriller_crime', 'genre_non-fiction', 'genre_poetry', 'genre_romance', 'genre_young-adult', 't0', 't1', 't2', 't3', 't4', 't5', 't6', 't7', 't8', 't9', 't10', 't11', 't12', 't13', 't14', 't15', 't16', 't17', 't18', 't19', 't20', 't21', 't22', 't23', 't24', 't25', 't26', 't27', 't28', 't29', 't30', 't31', 't32', 't33', 't34', 't35', 't36', 't37', 't38', 't39']


In [25]:
# Validation
assert user_taste[topic_cols].isna().sum().sum() == 0
assert user_taste[genre_cols].isna().sum().sum() == 0
print("✓ No NaN")
print(user_taste[genre_cols].sum(axis=1).describe())  # genre row sum ~1.0

✓ No NaN
count    2.193850e+05
mean     1.000000e+00
std      6.875194e-08
min      9.999998e-01
25%      9.999999e-01
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64


In [26]:
# Upload
import os, time
from huggingface_hub import HfApi

os.makedirs("/kaggle/working/phase3", exist_ok=True)

#F1
# user_taste.to_parquet("/kaggle/working/phase3/user_taste_profile.parquet", index=False)
#F2
local_path = "/kaggle/working/phase3/user_taste_profile_f5_binary_positive.parquet"
user_taste.to_parquet(local_path, index=False)

api = HfApi()

for attempt in range(3):
    try:
        api.upload_file(
            path_or_fileobj=local_path,
            # path_in_repo="phase3/user_taste_profile.parquet",

            # F2 → đổi thành:
            # path_in_repo="phase3/user_taste_profile_f2_rating_only.parquet",
            
            # # F3:
            # path_in_repo="phase3/user_taste_profile_f3_rating_length.parquet",
            
            # # F4:
            # path_in_repo="phase3/user_taste_profile_f4_rating_votes_length.parquet",
            
            # # F5:
            path_in_repo="phase3/user_taste_profile_f5_binary_positive.parquet",


            repo_id=HF_REPO, repo_type="dataset", token=HF_TOKEN
        )
        print(f"✓ Uploaded! {len(user_taste):,} users × {user_taste.shape[1]-1} dims")
        break
    except Exception as e:
        print(f"Attempt {attempt+1} failed: {e}")
        time.sleep(5)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✓ Uploaded! 219,385 users × 50 dims
